In [0]:
# =========================================================
# Copy CSVs from Workspace -> Unity Catalog Volume
# =========================================================

WORKSPACE_DATA_DIR = "dbfs:/Workspace/Projects/Capstone/data"
VOLUME_SELECTED_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw"

dbutils.fs.mkdirs(VOLUME_SELECTED_DIR)

workspace_files = dbutils.fs.ls(WORKSPACE_DATA_DIR)

csv_files = [f for f in workspace_files if f.path.lower().endswith(".csv")]

print(f"Found {len(csv_files)} CSV files in Workspace")

for f in csv_files:
    filename = f.path.split("/")[-1]
    dest = f"{VOLUME_SELECTED_DIR}/{filename}"
    print(f"Copying {filename}")
    dbutils.fs.cp(f.path, dest, True)

print("\n=== Copy complete ===")
display(dbutils.fs.ls(VOLUME_SELECTED_DIR))


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
SELECTED_RAW_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw"
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership"

dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze")

print("SELECTED_RAW_DIR:", SELECTED_RAW_DIR)
print("BRONZE_DIR      :", BRONZE_DIR)

# ---------------------------------------------------------
# 1) Read CSVs (Unity Catalog Volume)
# ---------------------------------------------------------
print("\nReading CSV files...")
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(SELECTED_RAW_DIR))

# ---------------------------------------------------------
# 2) Get file path using Unity Catalog supported metadata
#    (instead of input_file_name())
# ---------------------------------------------------------
df = df.withColumn("source_file", F.col("_metadata.file_path"))

# Extract year and month from filename (YYYY-MM)
df = df.withColumn("year", F.regexp_extract("source_file", r"(20\d{2})-(\d{2})", 1).cast("int"))
df = df.withColumn("month", F.regexp_extract("source_file", r"(20\d{2})-(\d{2})", 2).cast("int"))

# ---------------------------------------------------------
# 3) Safety filter: Oct-2022 -> Sep-2024
# ---------------------------------------------------------
df = df.filter(
    ((F.col("year") == 2022) & (F.col("month").between(10, 12))) |
    ((F.col("year") == 2023) & (F.col("month").between(1, 12))) |
    ((F.col("year") == 2024) & (F.col("month").between(1, 9)))
)

# Optional: Check for any rows that failed extraction
bad = df.filter(F.col("year").isNull() | F.col("month").isNull()).count()
if bad > 0:
    print(f"Rows with missing year/month: {bad} (check filename format)")
else:
    print("year/month extracted successfully from _metadata.file_path")

total_rows = df.count()
print(f"\nTotal rows (Oct-2022..Sep-2024): {total_rows:,}")

# ---------------------------------------------------------
# 4) Write Bronze Parquet partitioned by year/month
# ---------------------------------------------------------
print("\nWriting Bronze Parquet (partitionBy year, month)...")

# Clean previous bronze output (optional)
try:
    dbutils.fs.rm(BRONZE_DIR, True)
except Exception:
    pass

(df.write
 .mode("overwrite")
 .partitionBy("year", "month")
 .parquet(BRONZE_DIR))

print("Bronze Parquet written to:", BRONZE_DIR)

# ---------------------------------------------------------
# 5) Validation: counts per month + expected 24 months
# ---------------------------------------------------------
print("\nPost-write validation...")

bronze_df = spark.read.parquet(BRONZE_DIR)

# Total rows from parquet
bronze_total = bronze_df.count()
print(f"Total rows in Bronze Parquet: {bronze_total:,}")

# Count rows per year
print("\n=== Records per year ===")
(bronze_df
 .groupBy("year")
 .agg(F.count("*").alias("rows"))
 .orderBy("year")
 .show(truncate=False))

# Count rows per year & month
print("\n=== Records per year & month ===")
month_counts = (bronze_df
                .groupBy("year", "month")
                .agg(F.count("*").alias("rows"))
                .orderBy("year", "month"))
month_counts.show(50, truncate=False)

# Distinct month partitions
distinct_months = bronze_df.select("year", "month").distinct().count()
print(f"\nDistinct (year, month) partitions: {distinct_months}")

if distinct_months == 24:
    print("Month coverage OK (24 months)")
else:
    print("Month coverage NOT OK (expected 24). Check missing files.")

print("\n=== DONE ===")
print("Downstream read example:")
print(f'df = spark.read.parquet("{BRONZE_DIR}")')


In [0]:
df.printSchema()